# Exploration of FD001 Dataset

## SECTION 1 — Project Overview

**Predictive Maintenance** utilizes historical sensor data to predict when equipment will fail.
**Remaining Useful Life (RUL)** is the amount of time (in cycles) an engine is expected to operate before failure.
**NASA C-MAPSS** dataset contains simulated turbofan engine degradation data over multiple operational cycles.
**Why FD001?** FD001 is the simplest subset with a single operating condition and a single fault mode (HPC degradation), making it ideal for establishing our data foundation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data.loader import load_subset

## SECTION 2 — Load FD001

In [ ]:
train_df, test_df, test_rul = load_subset("FD001")

## SECTION 3 — Dataset Shape

In [ ]:
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
print("Test RUL Shape:", test_rul.shape)

print(f"\nNumber of training engines: {train_df['unit'].nunique()}")
print(f"Number of test engines: {test_df['unit'].nunique()}")
print(f"Number of columns: {len(train_df.columns)}")

## SECTION 4 — Raw Data Inspection

In [ ]:
display(train_df.head())
display(test_df.head())
display(test_rul.head())

In [ ]:
train_df.info()

In [ ]:
test_df.info()

## SECTION 5 — Missing Values

In [ ]:
print("Missing values in Train:", train_df.isnull().sum().sum())
print("Missing values in Test:", test_df.isnull().sum().sum())
print("Missing values in Test RUL:", test_rul.isnull().sum().sum())

## SECTION 6 — Engine Lifetime

In [ ]:
engine_lifetimes = train_df.groupby("unit")["cycle"].agg(min_cycle="min", max_cycle="max", num_observations="count").reset_index()
display(engine_lifetimes.head())
display(engine_lifetimes["max_cycle"].describe())

plt.figure(figsize=(10, 6))
sns.histplot(engine_lifetimes["max_cycle"], bins=20, kde=True)
plt.title("Distribution of Final Training Cycles (Engine Lifetimes)")
plt.xlabel("Max Cycles (Lifetime)")
plt.ylabel("Count")
plt.show()

## SECTION 7 — Feature Statistics

In [ ]:
settings_cols = [c for c in train_df.columns if c.startswith("setting_")]
sensors_cols = [c for c in train_df.columns if c.startswith("sensor_")]

def calculate_stats(df, cols):
    stats = []
    for c in cols:
        stats.append({
            "feature": c,
            "variance": df[c].var(),
            "std_dev": df[c].std(),
            "unique_values": df[c].nunique()
        })
    return pd.DataFrame(stats).sort_values(by="variance", ascending=False)

print("\n--- Operational Settings ---")
display(calculate_stats(train_df, settings_cols))

print("\n--- Sensors ---")
display(calculate_stats(train_df, sensors_cols))

## SECTION 8 — Test RUL Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(test_rul["RUL"], bins=20, kde=True)
plt.title("Distribution of True Test RUL Values at Test-Series Cutoff")
plt.xlabel("Remaining Useful Life (Cycles)")
plt.ylabel("Count")
plt.show()

## SECTION 9 — Sensor Trajectories

In [ ]:
unit_id = 1
unit_df = train_df[train_df["unit"] == unit_id]

# Pick a few sensors that typically show degradation to visualize
sensors_to_plot = ["sensor_2", "sensor_3", "sensor_4", "sensor_7"]

fig, axes = plt.subplots(len(sensors_to_plot), 1, figsize=(10, 12), sharex=True)
fig.suptitle(f"Sensor Trajectories for Training Engine Unit {unit_id}", fontsize=16)

for i, sensor in enumerate(sensors_to_plot):
    axes[i].plot(unit_df["cycle"], unit_df[sensor])
    axes[i].set_ylabel(sensor)

axes[-1].set_xlabel("Cycle")
plt.tight_layout()
plt.show()

## SECTION 10 — Basic Observations

**OBSERVATION:** Several sensors have a variance of 0.0 (e.g., sensor_1, sensor_10, sensor_18, sensor_19).
**INTERPRETATION:** These sensors do not change over the entire operational lifetime of any engine in the FD001 dataset.
**DECISION:** Keep all features for now to ensure consistency across all subsets. Feature selection will be done later.

**OBSERVATION:** Missing values are 0 across train, test, and test_rul.
**INTERPRETATION:** Data is well-formatted and clean.
**DECISION:** No imputation is required.

**OBSERVATION:** Engine lifetimes range from 128 to 362 cycles, with an average around 206.
**INTERPRETATION:** Engines fail at different operating times, highlighting the importance of condition-based modeling instead of average-lifetime guessing.
**DECISION:** Use actual condition sensors for sequence modeling in future milestones.

## SECTION — TRAINING RUL LABEL ENGINEERING

**Why raw RUL is calculated this way:**
RUL is defined as the time (in cycles) remaining before an engine fails. For the training data, we observe each engine until failure. Thus, the RUL at any given cycle is simply the engine's final observed cycle minus the current cycle.

**Why the final training cycle has RUL = 0:**
At the final cycle, the engine has failed (or reached the end of its useful life). Therefore, it has 0 cycles of remaining life.

**Why early-life RUL clipping can be useful for supervised learning:**
In the early life of an engine, degradation is typically negligible or unobservable, meaning sensor readings remain relatively constant. A model trying to predict a very high RUL (e.g., 300 cycles) based on early-life data may struggle because the engine looks exactly the same as one with 250 cycles remaining. Clipping the RUL assumes a constant "healthy" state until degradation begins to manifest. This simplifies the learning task.

*Clipping is a modeling decision, not a change to the physical definition of RUL.*
* RUL is retained as the original target.
* RUL_clipped is a separate modeling target.

**Note:** 125 is our initial baseline cap and will be evaluated later.

In [ ]:
from src.data.rul import add_training_targets

train_with_targets = add_training_targets(train_df)
display(train_with_targets.head())

In [ ]:
import src.config as config

print("1. Every training engine reaches RUL = 0:", (train_with_targets.groupby("unit")["RUL"].min() == 0).all())
print("2. Minimum RUL is:", train_with_targets["RUL"].min())
print("3. Maximum RUL is:", train_with_targets["RUL"].max())
print("\n4. Distribution statistics for RUL:\n", train_with_targets["RUL"].describe())
print("\n5. Distribution statistics for RUL_clipped:\n", train_with_targets["RUL_clipped"].describe())

num_clipped = (train_with_targets["RUL"] > config.DEFAULT_RUL_CAP).sum()
pct_clipped = num_clipped / len(train_with_targets) * 100
print(f"\n6. Number of observations affected by clipping: {num_clipped}")
print(f"Percentage affected: {pct_clipped:.2f}%")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(train_with_targets["RUL"], bins=30, kde=True)
plt.title("Raw RUL Distribution (Training)")
plt.xlabel("Raw RUL (Cycles)")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(train_with_targets["RUL_clipped"], bins=30, kde=True, color="orange")
plt.title(f"Clipped RUL Distribution (Cap={config.DEFAULT_RUL_CAP})")
plt.xlabel("Clipped RUL (Cycles)")
plt.ylabel("Count")
plt.show()

unit_id = 1
unit_targets = train_with_targets[train_with_targets["unit"] == unit_id]

plt.figure(figsize=(10, 6))
plt.plot(unit_targets["cycle"], unit_targets["RUL"], label="Raw RUL", linestyle="--")
plt.plot(unit_targets["cycle"], unit_targets["RUL_clipped"], label="Clipped RUL", linewidth=2)
plt.title(f"RUL Trajectory for Engine {unit_id}")
plt.xlabel("Cycle")
plt.ylabel("Remaining Useful Life")
plt.legend()
plt.grid(True)
plt.show()